## Ensemble Approach

The overarching goal is to compare as many models as possible to the “truth” model. Set a range of soils/parameters to get many conductivity models and TDEM responses to compare to the “truth”. 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import phydrus as ps
import simpeg.electromagnetics.time_domain as tdem
from simpeg import maps
### HYDRUS-1D AUTOMATION WITH PYWINAUTO
from pathlib import Path
import shutil
import subprocess
from pywinauto import Desktop
import time
from pywinauto.keyboard import send_keys
import pyperclip
import itertools 

### 1. Define "Truth" Model
##### This will require a forward model process to get a result to compare to

### 1.1 Set Necessary Folder Connections

In [11]:
install_dir = Path(r"C:\Program Files (x86)\PC-Progress\Hydrus-1D 4.xx")
project_dir = Path(r"C:\TDEM HYDRUS\Drainage - SHARP")
runner_dir = Path(r"C:\TDEM HYDRUS\hydrus_runner")

selector_file = project_dir / "SELECTOR.IN"
solver_src = install_dir / "H1D_UNSC.EXE"
solver_dst = runner_dir / "H1D_UNSC.EXE"

project_file = r"C:\TDEM HYDRUS\Drainage - SHARP.h1d"

##### 1.1 Set van Genuchten Parameters for Known Soil Types

In [12]:
### set soil type parameter options for "truth" forward model

truth_soil_type = "sand"  ### options: custom, sand, loamy sand, sandy loam, loam, silt, silt loam, clay

### Custom option 
custom_vg_params = {
    "custom": {
    "theta_r": 0.05,
    "theta_s": 0.4,
    "alpha": 0.1,
    "n": 2.0,
    "Ks": 10.0,
    "l": 0.5
}}

### OR preset values for sand, loamy sand, sandy loam, loam, silt, silt loam, clay

vg_presets = {
    "sand": {
        "theta_r": 0.045,
        "theta_s": 0.43,
        "alpha": 0.145,
        "n": 2.68,
        "Ks": 29.7, ### units of cm/hr
        "l": 0.5
    },
    "loamy sand": {
        "theta_r": 0.057,
        "theta_s": 0.41,
        "alpha": 0.124,
        "n": 2.28,
        "Ks": 14.5917, ### units of cm/hr
        "l": 0.5
    },
    "sandy loam": {
        "theta_r": 0.065,
        "theta_s": 0.41,
        "alpha": 0.075,
        "n": 1.89,
        "Ks": 4.42083, ### units of cm/hr
        "l": 0.5
    },
    "loam": {
        "theta_r": 0.078,
        "theta_s": 0.43,
        "alpha": 0.036,
        "n": 1.56,
        "Ks": 1.04, ### units of cm/hr
        "l": 0.5
    },
    "silt": {
        "theta_r": 0.034,
        "theta_s": 0.46,
        "alpha": 0.016,
        "n": 1.37,
        "Ks": 0.25, ### units of cm/hr
        "l": 0.5
    },
    "silt loam": {
        "theta_r": 0.067,
        "theta_s": 0.45,
        "alpha": 0.02,
        "n": 1.41,
        "Ks": 0.45,
        "l": 0.5
    },
    "clay": {
        "theta_r": 0.068,
        "theta_s": 0.38,
        "alpha": 0.008,
        "n": 1.09,
        "Ks": 0.2, ### units of cm/hr
        "l": 0.5
    }
}

### 1.2 Choose Parameters for "Truth"
##### Custom values OR sand, loamy sand, sandy loam, loam, silt, silt loam, clay

In [13]:
def truth_vg_params(soil_type):
    truth_soil_type = soil_type.lower().strip()
    if truth_soil_type in vg_presets:
        return vg_presets[truth_soil_type]
    elif truth_soil_type == "custom":
        return custom_vg_params["custom"]
    else:
        print(f"Warning: Unrecognized soil type '{soil_type}'. Defaulting to 'loam' parameters.")
        return vg_presets["loam"]

In [14]:
truth_vg_params = truth_vg_params(
    soil_type=truth_soil_type,
    )

print(truth_vg_params)

{'theta_r': 0.045, 'theta_s': 0.43, 'alpha': 0.145, 'n': 2.68, 'Ks': 29.7, 'l': 0.5}


### 1.3 Run HYDRUS for "Truth" Model

In [ ]:
### One function to automate the HYDRUS run process for future use
### Opens HYDRUS, loads the project file, and executes the run

def run_hydrus(hydrus_gui, project_file, vg_params, wait_after_run=10):

    hydrus_gui = Path(hydrus_gui)
    project_file = Path(project_file)
    ### Update HYDRUS files 
    ## Assign new parameters to the SELECTOR.IN file

    new_line = (
        f"{vg_params['theta_r']:>8.3f}"
        f"{vg_params['theta_s']:>8.3f}"
        f"{vg_params['alpha']:>8.3f}"
        f"{vg_params['n']:>8.3f}"
        f"{vg_params['Ks']:>12.4f}"
        f"{vg_params['l']:>8.3f}"
    )

    text = selector_file.read_text()
    lines = text.splitlines()

    for i, line in enumerate(lines):
        if "thr" in line and "ths" in line and "Alfa" in line:
            lines[i + 1] = new_line
            break

    selector_file.write_text("\n".join(lines))

    print("Updated HYDRUS parameters:")
    print(new_line)

    ### Open HYDRUS GUI and detect window
    process = subprocess.Popen(
        [str(hydrus_gui)],
        cwd=hydrus_gui.parent
    )

    time.sleep(wait_after_run) # give HYDRUS time to open

    ### Find the HYDRUS window
    def get_hydrus_window():
        windows = Desktop(backend="uia").windows()

        for w in windows:
            title = w.window_text()

        # Select only the actual HYDRUS program window
            if title == "HYDRUS-1D" or title.startswith("HYDRUS-1D -"):
                return w

        raise RuntimeError("HYDRUS window not found.")

    
    ### Open HYDRUS and file window
    hydrus_window = get_hydrus_window()
    hydrus_window.set_focus()
    time.sleep(1)

    print("Focused window:", hydrus_window.window_text())

    send_keys("%f")
    send_keys("o")

    ### Paste file name and open project
    hydrus_app = Desktop(backend="uia").window(title_re=".*HYDRUS-1D.*")
    open_dialog = hydrus_app.child_window(title="Open", control_type="Window")

    open_dialog.set_focus()
    time.sleep(0.5)

    pyperclip.copy(project_file)

    send_keys("%n")      # focus File name box
    time.sleep(0.5)

    send_keys("^v")      # paste path
    time.sleep(0.5)

    send_keys("{ENTER}") # open
    time.sleep(2)

    ### Open calculation menu and execute run
    hydrus_app = Desktop(backend="uia").window(title_re=".*HYDRUS-1D - Drainage - SHARP.*")
    hydrus_app.set_focus()
    time.sleep(0.5)

    send_keys("%c")
    time.sleep(0.5)

    items = hydrus_app.descendants(title="Execute HYDRUS", control_type="MenuItem")

    print(len(items))
    for item in items:
        print(item)

    items[0].click_input()

    ### Hit okay to start the run 
    time.sleep(0.5)
    send_keys("{ENTER}")

    ### Wait and enter to finish
    time.sleep(wait_after_run)
    send_keys("{ENTER}")

    return process

In [17]:
run_hydrus(
    hydrus_gui=Path(r"C:\Program Files (x86)\PC-Progress\Hydrus-1D 4.xx\Hydrus1D.exe"),
    project_file=Path(r"C:\TDEM HYDRUS\Drainage - SHARP.h1d"), truth_vg_params=truth_vg_params,
    wait_after_run=10
)

Updated HYDRUS parameters:
   0.045   0.430   0.145   2.680     29.7000   0.500
Focused window: HYDRUS-1D - Drainage - SHARP
2
uia_controls.MenuItemWrapper - 'Execute HYDRUS', MenuItem
uia_controls.MenuItemWrapper - 'Execute HYDRUS', MenuItem


<Popen: returncode: None args: ['C:\\Program Files (x86)\\PC-Progress\\Hydru...>

### 2. Add Error

### 3. Build/Run Ensemble

##### 3.1 Set Parameter Ranges to Create Scenarios for Ensemble Run

In [ ]:
### Generate parameter sets/range for ensemble runs
theta_r_values = [0.045, 0.057, 0.065, 0.078, 0.034, 0.067, 0.068]
theta_s_values = [0.43, 0.41, 0.41, 0.43, 0.46, 0.45, 0.38]
alpha_values = [0.145, 0.124, 0.075, 0.036, 0.016, 0.02, 0.008]
n_values = [2.68, 2.28, 1.89, 1.56, 1.37, 1.41, 1.09]
Ks_values = [29.7, 14.5917, 4.42083, 1.04, 0.25, 0.45, 0.2]
l_values = [0.5]

##### 3.2 Loop through each scenario for varrying parameters
##### Edit fixed and changing parameters

In [ ]:
results = []
scenario_counter = 1

### change the variables in for loop dependent on what is fixed vs. what is varied in the scenarios!!!!

for theta_s, alpha, n, Ks in itertools.product(     ### THIS LOOPS THROUGH THE RANGE SET ABOVE [EVERY COMBINATION OF THE PARAMETERS]
        theta_s_values,
        alpha_values,
        n_values,
        Ks_values):
    
    scenario_name = f"Scenario_{scenario_counter:03d}"

### Values are fixed at "truth" parameters or varied across the range depending on the scenario design - change as needed

    vg_ensemble = {
        "theta_r": truth_vg_params["theta_r"],  ### Fixed
        "theta_s": truth_vg_params["theta_s"],  ### Fixed
        "alpha": truth_vg_params["alpha"],   ### Fixed
        "n": n,            ### Varies
        "Ks": Ks,           ### Varies
        "l": 0.5            ### Fixed
    }

    print(f"Running scenario {scenario_counter}: n={n}, Ks={Ks}")
    
### Call the function to run HYDRUS with specified VG parameters for each scenario

    run_hydrus(
        hydrus_gui=Path(r"C:\Program Files (x86)\PC-Progress\Hydrus-1D 4.xx\Hydrus1D.exe"),
        project_file=Path(r"C:\TDEM HYDRUS\Drainage - SHARP.h1d"),
        vg_ensemble=vg_ensemble,
        wait_after_run=10
    )

    scenario_counter += 1

results.append({
        "scenario": scenario_name,
        "theta_r": vg_ensemble["theta_r"],
        "theta_s": vg_ensemble["theta_s"],
        "alpha": vg_ensemble["alpha"],
        "n": vg_ensemble["n"],
        "Ks": vg_ensemble["Ks"],
        "l": vg_ensemble["l"],
        # "objective": objective_score  # add later after comparison step
    })

results_df = pd.DataFrame(results)

results_df.to_csv(
    "ensemble_summary.csv",
    index=False
)

### 4. Pull Results from HYDRUS

### 5. Compare "Truth" to Predicted

### 6. Interpret Results